# Solution to p09

Candidates are stably ordered by decreasing score with original index as the tie-break. Suppression compares only candidates of the selected candidate's class, and equality at the IoU threshold is retained because suppression is strictly above the threshold.

In [ ]:
import torch

boxes = torch.tensor([[0.,0.,4.,4.], [.5,0.,4.5,4.], [0.,0.,4.,4.], [5.,5.,7.,7.]], dtype=torch.float32)
scores = torch.tensor([.90, .80, .85, .20], dtype=torch.float32)
labels = torch.tensor([0, 0, 1, 0], dtype=torch.int64)
score_threshold = 0.30
iou_threshold = 0.50
ATOL = 1e-6
RTOL = 1e-6

def pairwise_iou(boxes_a, boxes_b):
    if boxes_a.ndim != 2 or boxes_b.ndim != 2 or boxes_a.shape[1] != 4 or boxes_b.shape[1] != 4:
        raise ValueError("boxes must be matrices with four columns")
    left_top = torch.maximum(boxes_a[:,None,:2], boxes_b[None,:,:2])
    right_bottom = torch.minimum(boxes_a[:,None,2:], boxes_b[None,:,2:])
    intersection = (right_bottom-left_top).clamp_min(0).prod(-1)
    area_a = (boxes_a[:,2:]-boxes_a[:,:2]).prod(-1)
    area_b = (boxes_b[:,2:]-boxes_b[:,:2]).prod(-1)
    return intersection / (area_a[:,None]+area_b[None,:]-intersection)

def class_aware_nms(boxes, scores, labels, score_threshold=0.30, iou_threshold=0.50):
    if boxes.ndim != 2 or boxes.shape[1] != 4 or scores.shape != (boxes.shape[0],) or labels.shape != (boxes.shape[0],):
        raise ValueError("shape mismatch")
    if boxes.dtype != torch.float32 or scores.dtype != torch.float32 or labels.dtype != torch.int64:
        raise ValueError("dtype mismatch")
    eligible = [i for i in range(boxes.shape[0]) if float(scores[i]) >= score_threshold]
    pending = sorted(eligible, key=lambda i: (-float(scores[i]), i))
    kept = []
    while pending:
        selected = pending.pop(0)
        kept.append(selected)
        survivors = []
        for candidate in pending:
            same_class = labels[candidate].item() == labels[selected].item()
            overlap = pairwise_iou(boxes[selected:selected+1], boxes[candidate:candidate+1])[0,0].item()
            if not (same_class and overlap > iou_threshold):
                survivors.append(candidate)
        pending = survivors
    return torch.tensor(kept, dtype=torch.int64)

kept = class_aware_nms(boxes, scores, labels, score_threshold, iou_threshold)


### Answer check

In [ ]:
assert kept.dtype == torch.int64
assert kept.tolist() == [0,2]
assert pairwise_iou(boxes[0:1], boxes[1:2]).item() > 0.5
assert labels[0].item() != labels[2].item()
